<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

## Improve a Tour with 2-opt Local Search

**Reverse a segment of a complete tour and accept only a strictly shorter candidate.**

Keep the five-city symmetric distance matrix, the requirement to visit every city once and return, and the total-distance objective. The decision is a discrete visit order. The clock observations, calibration equality, and correction bound do not apply to this delivery system. The search begins from the feasible tour $(1,0,4,3,2,1)$ with distance 22. The new operation revises a completed tour rather than extending a partial route.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 1 · Choose a complete tour

A vehicle must visit cities $0,1,2,3,4$ once each and return to its starting city. A **tour** is that complete closed route. Shorter total distance means less travel.

Let $C_{ij}$ be the fixed travel distance from city $i$ to city $j$. The source gives no physical unit, so we report **distance units**. All city pairs are connected, and the distances are symmetric: $C_{ij}=C_{ji}$. The numbers are travel costs; we do not assume they are straight-line distances on a map.

| From / to | $0$ | $1$ | $2$ | $3$ | $4$ |
|:---|---:|---:|---:|---:|---:|
| $0$ | 0 | 2 | 9 | 10 | 7 |
| $1$ | 2 | 0 | 8 | 5 | 6 |
| $2$ | 9 | 8 | 0 | 4 | 3 |
| $3$ | 10 | 5 | 4 | 0 | 1 |
| $4$ | 7 | 6 | 3 | 1 | 0 |

Write the decision as $x=(v_0,v_1,v_2,v_3,v_4,v_5)$, where $v_k$ is the city visited at position $k$. Feasibility requires that $v_0,\ldots,v_4$ contain every city exactly once and $v_5=v_0$. The objective is

$$
f(x)=\sum_{k=0}^{4} C_{v_k,v_{k+1}}.
$$

The final term includes the return trip. For example,

$$
f(2,4,3,1,0,2)=3+1+5+2+9=20.
$$

The distance matrix is fixed data. The visit order is the physical decision. Here the raw performance output, total distance, is also the scalar objective. A route with a missing city or return trip is rejected before any score comparison.


In [ ]:
import numpy as np

# Fixed travel distances, copied from the supplied example_1.py
DISTANCES = np.array([
    [0, 2, 9, 10, 7],
    [2, 0, 8, 5, 6],
    [9, 8, 0, 4, 3],
    [10, 5, 4, 0, 1],
    [7, 6, 3, 1, 0],
], dtype=int)
CITY_COUNT = len(DISTANCES)


def evaluate_tour(tour):
    tour = tuple(tour)
    valid_labels = all(isinstance(city, (int, np.integer))
                       and 0 <= city < CITY_COUNT for city in tour)
    feasible = (len(tour) == CITY_COUNT + 1 and valid_labels
                and tour[0] == tour[-1]
                and set(tour[:-1]) == set(range(CITY_COUNT)))
    if not feasible:
        return {"decision": tour, "feasible": False,
                "edge_distances": None, "objective": None}
    edges = np.array([DISTANCES[a, b] for a, b in zip(tour, tour[1:])])
    return {"decision": tour, "feasible": True,
            "edge_distances": edges, "objective": int(edges.sum())}


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 2 · Define one local change: 2-opt

Greedy construction extends an unfinished route. **Local search** starts with a complete feasible tour and repeatedly considers small changes to it. Its **neighborhood**, $N(x)$, is the set of tours reachable from the current tour $x$ by one allowed change.

A **2-opt move** removes two nonadjacent edges and reconnects the pieces into a single cycle. For our symmetric distances, this is implemented by reversing a segment of the city order.

Compare the two orange city tiles in the rows below. Reversing their order keeps every city in the tour, but changes the two connections at the segment boundaries. The small numbers between city tiles are edge distances.

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/05-1_heuristic_optimization_greedy_and_local_search/assets/two_opt_segment_reversal-v3.png" alt="Before 2-opt, the tour 1, 0, 4, 3, 2, 1 has edge distances 2, 7, 1, 4, 8 and total 22. Reversing cities 3 and 2 gives tour 1, 0, 4, 2, 3, 1 with edge distances 2, 7, 3, 4, 5 and total 21." width="512" style="display: block; max-width: 100%; height: auto; margin: 0; background-color: #fff;">
</div>

Starting from the length-22 tour, reverse the segment $(3,2)$:

$$
\begin{aligned}
x&=(1,0,4,3,2,1),& f(x)&=22,\\
x'&=(1,0,4,2,3,1),& f(x')&=21.
\end{aligned}
$$

The move removes edges $(4,3)$ and $(2,1)$, with total distance $1+8=9$, and adds $(4,2)$ and $(3,1)$, with total $3+5=8$. The reversed internal edge has the same distance in either direction. Thus the total decreases by 1.

In the code, positions $i$ through $j$ are reversed, with $1\le i<j\le n-1$ and $n=5$. The first and final city stay fixed. Reversing all four internal positions only reverses the same undirected cycle, so that redundant move is excluded. The five remaining moves preserve feasibility. The indices are positions in the tour, not city labels.


In [ ]:
def two_opt_neighbors(tour):
    tour = tuple(tour)
    city_count = len(tour) - 1
    for i in range(1, city_count - 1):
        for j in range(i + 1, city_count):
            if i == 1 and j == city_count - 1:
                continue  # Same undirected cycle, written in reverse.
            candidate = tour[:i] + tour[i:j + 1][::-1] + tour[j + 1:]
            yield (i, j), candidate


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

The initial tour is $x=(1,0,4,3,2,1)$, the nearest-neighbor result from city 1. The following scan evaluates each complete 2-opt neighbor and compares its distance with 22 after checking feasibility.


In [ ]:
initial_tour = (1, 0, 4, 3, 2, 1)
initial_result = evaluate_tour(initial_tour)
for move, tour in two_opt_neighbors(initial_tour):
    result = evaluate_tour(tour)
    if result["feasible"]:
        change = result["objective"] - initial_result["objective"]
        print(f"Reverse positions {move}: {tour} | distance {result['objective']}"
              f" | change {change:+d}")


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 3 · Repeat strictly improving moves

Use **best improvement**: evaluate every neighbor and choose the one with the smallest distance. Accept it only if its distance is strictly smaller than the current value. Ties between equally good improving moves follow the order in which the code lists them.

Repeat this calculation from the accepted tour. If a full neighborhood contains no strict improvement, stop. The returned tour $x_{\mathrm{local}}$ satisfies

$$
f(x_{\mathrm{local}})\le f(x')
\quad\text{for every }x'\in N(x_{\mathrm{local}}).
$$

This is local optimality **with respect to the chosen neighborhood**. It is not the differentiable stationary-point condition from the clock problem. The city order is discrete, and the allowed moves define what “nearby” means.

Strict improvement prevents revisiting a tour. Since this problem has finitely many tours, the search terminates. Accepting equal-cost moves without another rule could cause cycling. A run stopped early by a search budget would not establish the neighborhood condition above.


In [ ]:
def local_search(initial_tour, neighborhood=two_opt_neighbors):
    current = evaluate_tour(initial_tour)
    if not current["feasible"]:
        raise ValueError("Local search needs a complete feasible initial tour.")
    history = [current]
    accepted_moves = []
    neighbor_evaluations = 0
    while True:
        best = current
        best_move = None
        for move, tour in neighborhood(current["decision"]):
            candidate = evaluate_tour(tour)
            neighbor_evaluations += 1
            if not candidate["feasible"]:
                continue
            if candidate["objective"] < best["objective"]:
                best, best_move = candidate, move
        if best_move is None:
            return {"result": current, "history": history,
                    "accepted_moves": accepted_moves,
                    "neighbor_evaluations": neighbor_evaluations,
                    "stop_reason": "No strictly improving neighbor"}
        current = best
        history.append(current)
        accepted_moves.append(best_move)


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

For the start-1 greedy tour, best improvement gives distances $22\to21\to20$. The final scan checks every neighbor once more before stopping. The final scan establishes local optimality for 2-opt. A global claim would additionally require covering all feasible tours or another independent proof.

Read each column from the orange current distance to its five neighbor distances. Teal marks an accepted improvement. The last column still checks all five neighbors; its smallest value is 21, so it cannot improve the current 20. Every value is in distance units.

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/05-1_heuristic_optimization_greedy_and_local_search/assets/local_search_complete_scans.png" alt="Scan 1 evaluates neighbors 35, 30, 24, 22, and 21 from current distance 22 and accepts 21. Scan 2 evaluates 31, 33, 20, 25, and 22 from 21 and accepts 20. Scan 3 evaluates 30, 33, 21, 24, and 22 from 20 and stops because no neighbor improves it." width="512" style="display: block; max-width: 100%; height: auto; margin: 0; background-color: #fff;">
</div>


In [ ]:
improved_run = local_search(initial_tour)
for step, result in enumerate(improved_run["history"]):
    print(f"Accepted moves {step}: {result['decision']}"
          f" | distance {result['objective']}")
print("Reversed position pairs:", improved_run["accepted_moves"])
print("Neighbor evaluations, including the final scan:",
      improved_run["neighbor_evaluations"])
print("Stop:", improved_run["stop_reason"])


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 4 · Source material and method reference

The supplied `Lecture_10_1_Heuristic_Optimization/10_1.pptx` introduces heuristic construction rules. Its `ExampleCode_10_1/example_1.py` supplies the five-city distance matrix, nearest-neighbor rule, and original start at city 2. This notebook adds the 2-opt neighborhood, best-improvement updates, and the complete final scan.

Michael A. Trick's [Heuristics](https://mat.tepper.cmu.edu/orclass/integer/node17.html) describes nearest-neighbor tour construction and two-opt tour improvement. The worked calculations use the supplied matrix.
